# intraday-vol-of-vol-v1

BigAlpha 2026 · AI 因子挖掘 (AI track) submission.

- source: `platform/submission/intraday_vol_of_vol_v1_main.py`
- sha1: `31d2d2300063f488352fb57805e923d5e35862ce`
- generated by `platform/submission/build_notebook.py` — the code cell below is a **verbatim copy** of that committed file, so this notebook contains the code that is in git.

AI-track note: the derivation chain for this factor (prompts, agent sessions, where the work departed from its source) is in `factors/intraday-vol-of-vol-v1/provenance.md`, and the AI application document is submitted alongside as markdown.


In [ ]:
"""intraday-vol-of-vol-v1 — the submission `main()`. Submitted DESPITE a local kill, on purpose.

    brv    = SUM(ret*ret) over the minutes of one intraday BLOCK        (8 blocks per day)
    cv     = stddev(brv) / mean(brv)                                    per (day, instrument)
    factor = -z_by_day(cv)                                              higher = better

Within-day *dispersion of volatility*: not how volatile a name was, but how unevenly that volatility
arrived. A name whose variance is concentrated in one block was moved by an event; a name whose
variance is spread evenly was moved by a crowd. The ex-ante sign is **+** on the submitted object
(negated internally), pre-registered before data contact in `factors/intraday-vol-of-vol-v1/spec.md`.

## Why this is being submitted after being RETIRED, stated plainly

k23 retired it on a hand-rolled residual gate; k24 confirmed with the official `factor_pool` B-proxy:
**ModelScore 1.2467, rank 10 of 11** against the organizers' 8 base factors + our two live formulas.
That kill stands *as a nomination decision* and this file does not overturn it by argument.

What changed is the recognition that **the kill was applied to the wrong decision.** The ~2.4 ModelScore
bar exists because the private board takes **≤ 2** nominated factors, so a new factor only matters if it
DISPLACES an incumbent. Submissions are not scarce in the same way — 47 of 50 remain — and each one buys
a real public-board reading. Using a displacement threshold to gate a submission spends a scarce
resource (information) to protect an abundant one (slots).

And the proxy is measurably not the thing it proxies for. **B is scored inside one global Elastic Net
over ALL teams' factors**, not against the base library. k24 put `tsr` at rank 2 and `esr_v2` at rank 3
against the base library *in the same week our public B fell 0.6286 → 0.3295 at rising A*. So the base
library ranks our factors highly while the real pool erodes them: the two disagree, and we had already
measured that they disagree before this factor was killed on the one that is not the target.

## Why THIS factor, of the retired ones

1. **It is the only retired object whose kill was purely a pool-ranking**, not a sign inversion, not a
   leg beating the composite, and not a contract failure.
2. **Its contract profile is the best in the repo.** Per-day coverage mean *and* min both **1.000000**
   over the 225-day 2023 holdout, and **1.000000 on 2020-02-03** — the mass-limit day that invalidated
   slot 2. 29 teams sit at `-2.00000`; this construction cannot land there on coverage.
3. **Out-of-sample raw signal survived almost intact**: `ic_ir` decayed **4.8%** discovery→holdout
   (t 5.6309 on 225 days). What decayed 23% was the *orthogonal residual* — which is exactly the
   quantity a local proxy estimates badly and the real board measures directly.
4. **It is not the crowded wiki object.** Measured `|corr|` **0.1201** out-of-sample against 高频偏度
   (`rskew`) and 下行波动占比 (`rsj`), the two volatility-shape factors on BigQuant's own published
   minute-bar list — and that number *improved* from 0.2348 on discovery.

The honest counter-case, recorded because it may well be right: `volatility_5` is the single most
weight-stable factor in the base library (2.9765, rank 1), and if the teams' pool is similarly saturated
on the volatility axis this submission scores near zero on B. **That is the question being bought.**

## Differences from the measured k22/k23 scan, all of them contract-motivated

1. **No `bigalpha_2026_instruments` join.** k22 joined it; a submission uses `datasources` tables only.
   `z_by_day` is a per-day *linear* transform, so it is rank-preserving: widening the cross-section
   changes the z values but not the within-day ordering of any name, and the evaluator scores its own
   universe cross-sectionally. So the wider panel cannot change this factor's score, and it removes a
   join and a table dependency.
2. **The comparator half of the scan is gone.** `rskew`, `rsj`, `obi5`, `amount`/`volume`/`deal_number`
   sums existed to measure K-CROWD and the legs. A submission needs the factor only, so the second
   aggregate branch and the level-5 volume columns are dropped — strictly less compute, same factor.
3. **A zero-dispersion day maps to 0.0 rather than NaN.** k22's `z_by_day` divides by the day's
   cross-sectional `std` with `.replace(0, np.nan)`, so a day where every name shared one `cv` would
   null the entire cross-section and *delete the day* — the exact rule-2 violation that killed slot 2.
   Measured coverage min was 1.000000 on both windows, so this branch was never taken and **cannot
   change any measured value**; it exists so an unmeasured day cannot invalidate the submission.

## Choices carried over from the measured construction, unchanged

- **Blocks are cut to respect the lunch break**, not to be equal-length: ≤1000, ≤1030, ≤1100, ≤1130,
  ≤1330, ≤1400, ≤1430, else. The 11:30→13:00 gap is not a 90-minute bar of zero volatility.
- **`hhmm` 931..1456.** 09:31 is the first minute with a within-session `open`; 14:57–15:00 closing
  auction bars stay out — an auction print is not a continuous-session return.
- **`stddev_samp` with a guarded fallback.** k22 measured the two forms agreeing to a median |diff| far
  below scale; `sqrt` is taken on a `clip(lower=0)` argument because `sqrt(negative)` RAISES
  `OutOfRangeException` on this platform rather than returning NaN.
- **`cv` fillna(0.0)** — the pre-registered fallback: a day with no variance genuinely has no dispersion
  of variance. This is what makes coverage 1.000000 rather than merely high.
- **PIT**: minutes of day *t* only, all aggregation within `(instrument, day)`. No cross-day window, no
  warmup needed, nothing from *t+1* touched.
- **No assert/raise anywhere**, and the end-bound is repaired rather than trusted (trap D1: a DAI date
  filter whose end carries no time resolves to 00:00 and silently drops the final day's bars).
"""


def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    HHMM_LO, HHMM_HI = 931, 1456

    SQL_TEMPLATE = """
    WITH r AS (
        SELECT
            instrument,
            CAST(date AS DATE) AS d,
            EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) AS hhmm,
            CASE WHEN open > 0 THEN close / open - 1 END AS ret
        FROM {bar1m}
    ),
    m AS (
        SELECT
            r.*,
            -- eight blocks, cut to respect the lunch break rather than to be equal-length
            CASE WHEN hhmm <= 1000 THEN 1 WHEN hhmm <= 1030 THEN 2
                 WHEN hhmm <= 1100 THEN 3 WHEN hhmm <= 1130 THEN 4
                 WHEN hhmm <= 1330 THEN 5 WHEN hhmm <= 1400 THEN 6
                 WHEN hhmm <= 1430 THEN 7 ELSE 8 END AS blk
        FROM r
        WHERE hhmm BETWEEN {lo} AND {hi}
    ),
    blocks AS (
        -- INNER level: realized variance per block
        SELECT d, instrument, blk,
               SUM(ret * ret) AS brv
        FROM m
        GROUP BY d, instrument, blk
    )
    -- OUTER level: dispersion of block RV across the day, both forms
    SELECT d AS date, instrument,
           AVG(brv)         AS rv_mean,
           stddev_samp(brv) AS rv_sd_primary,
           AVG(brv * brv)   AS rv_sq_mean
    FROM blocks
    GROUP BY d, instrument
    """
    sql = SQL_TEMPLATE.format(bar1m=datasources["bar1m"], lo=HHMM_LO, hi=HHMM_HI)

    end = pd.Timestamp(end_date)
    if (end.hour, end.minute, end.second) == (0, 0, 0):
        end = end + pd.Timedelta(hours=23, minutes=59, seconds=59)
    filters = {"date": [str(pd.Timestamp(start_date)), str(end)]}

    df = dai.query(sql, filters=filters, compression=True).df()
    df["date"] = pd.to_datetime(df["date"])          # DAI returns datetime.date (trap D4)

    num = lambda c: pd.to_numeric(df[c], errors="coerce").astype("float64")   # noqa: E731
    rv_mean, rv_sq_mean = num("rv_mean"), num("rv_sq_mean")
    rv_sd_primary = num("rv_sd_primary")

    # Guarded fallback: sqrt(negative) RAISES on this platform, so clip before the root. Population
    # vs sample (n/(n-1)) makes the two forms differ slightly; k22 measured that gap far below scale.
    rv_sd_fallback = np.sqrt((rv_sq_mean - rv_mean ** 2).clip(lower=0.0))
    rv_sd = rv_sd_primary.where(rv_sd_primary.notna(), rv_sd_fallback)

    # PRE-REGISTERED fallback: a day with no variance genuinely has no dispersion of variance.
    cv = (rv_sd / rv_mean.where(rv_mean > 0)).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # Cross-sectional z, per day. Linear per day, therefore rank-preserving. A day of zero
    # cross-sectional dispersion yields 0.0 rather than NaN, so no day can be emptied (rule 2).
    day = df["date"]
    g = cv.groupby(day)
    z = (cv - g.transform("mean")) / g.transform("std")
    factor = -z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    factor = factor.where(np.isfinite(factor))

    out = pd.DataFrame({
        "date": day,
        "instrument": df["instrument"],
        "factor": factor.astype("float64"),
    })
    out = out.drop_duplicates(subset=["date", "instrument"], keep="first")

    # No whole day is ever removed, which rule 2 forbids; individually-NaN rows are dropped, which
    # the 40%-per-day allowance permits. Measured coverage min was 1.000000 on both windows.
    return out.dropna(subset=["factor"]).sort_values(["date", "instrument"]).reset_index(drop=True)


In [ ]:
# The evaluation-module call must live in the notebook (competition requirement).
datasources = {
    'bar1m': 'bigalpha_2026_stock_bar1m',
    'financial': 'bigalpha_2026_financial',
}
START, END = '2019-06-05', '2023-12-31 23:59:59'

factor_data = main(datasources, START, END)
print('rows', len(factor_data), 'cols', list(factor_data.columns))
print('days', factor_data['date'].nunique(),
      'names', factor_data['instrument'].nunique())

try:
    from bigmodule import M
    result = M.bigalpha_eval._latest(factor_data=factor_data)
    print(dict(result.factor_analyze))
except Exception as exc:
    # Never raise from a submitted notebook: the executor returns no error detail, so a
    # raise is indistinguishable from a broken factor. Report and let the platform's own
    # scoring pass judge the returned frame.
    print('evaluator not run in this environment:', type(exc).__name__, exc)
